# 🛡️ Gradient Clipping & Pretraining Stability Lab

Welcome! In this notebook, we master <b>Gradient Clipping</b>—the essential safety circuit-breaker that prevents foundation models from crashing into NaNs.

---

## 1. The Metaphor: The Electrical Circuit Breaker ⚡

During pretraining across billions of tokens, your model will eventually encounter a bizarre, corrupted text file.<br>
When backpropagation runs on this rogue batch, gradients can suddenly surge from normal values (0.1) to <b>8,500.0</b>!
<br><br>
<ul>
  <li><b>Without Protection:</b> The optimizer multiplies those massive gradients into the weights. The weights explode to infinity (<code>NaN</code>), and your 3-month pretraining run is permanently ruined!</li>
  <li><b>With Gradient Clipping (The Circuit Breaker):</b> If the total length (norm) of the gradient exceeds a safety threshold (e.g. 1.0), it automatically scales the vector down to 1.0 while preserving its exact direction!</li>
</ul>

---
## 2. Visual Geometry: Scale Down, Preserve Direction

<div style='text-align: center; margin: 20px 0;'>
  <img src='assets/grad_clipping.png' width='850' style='border-radius: 8px; border: 1px solid #30363d;' alt='Gradient Clipping Geometry'>
</div>

---
## 3. Exercise 1: Pencil & Paper Calculation 📝

Suppose our model produces a dangerous gradient vector with 2 numbers:
<pre style='font-size: 15px; background: #161b22; color: #f85149; padding: 12px; border-radius: 8px;'>
g = [ 3.0,  4.0 ]
</pre>

Let our safety limit be: <b>max_norm = 1.0</b>.

### Step-by-Step on Paper:
1. <b>Calculate the total length (L2 norm) of the gradient:</b><br>
   Norm G = sqrt( 3.0² + 4.0² ) = sqrt( 9.0 + 16.0 ) = sqrt( 25.0 ) = <b>5.0</b>
<br><br>
2. <b>Check if it exceeds the limit:</b><br>
   Is 5.0 > 1.0? <b>YES! (A 5x electrical surge!)</b>
<br><br>
3. <b>Calculate the scaling multiplier:</b><br>
   Scale Factor = max_norm / G = 1.0 / 5.0 = <b>0.2</b>
<br><br>
4. <b>Multiply the gradient by 0.2:</b><br>
   - g'₁ = 3.0 · 0.2 = <b>0.6</b>
   - g'₂ = 4.0 · 0.2 = <b>0.8</b>

<br>
<b>Clipped Safe Gradient:</b> [ 0.6,  0.8 ]<br>
<i>Check the new norm: sqrt(0.6² + 0.8²) = sqrt(0.36 + 0.64) = 1.0 (Exactly 1.0!)</i>

---
## 4. Verify in PyTorch with Real Model Gradients

Run this cell to see PyTorch's <code>torch.nn.utils.clip_grad_norm_</code> perform this exact calculation on a model parameter:

In [1]:
import torch
import torch.nn as nn

# Create a dummy parameter and assign our dangerous [3.0, 4.0] gradient
param = nn.Parameter(torch.zeros(2))
param.grad = torch.tensor([3.0, 4.0])

print(f"Raw Gradient Vector:         {param.grad.tolist()}")
raw_norm = param.grad.norm(2).item()
print(f"Raw Gradient Norm (Length):  {raw_norm:.4f} (Exceeds safety limit of 1.0!)")

# Apply PyTorch's native gradient clipping
total_norm = torch.nn.utils.clip_grad_norm_([param], max_norm=1.0)

print(f"\nClipped Gradient Vector:     {[round(v, 4) for v in param.grad.tolist()]}")
clipped_norm = param.grad.norm(2).item()
print(f"New Gradient Norm (Length):  {clipped_norm:.4f} (Tamed to exactly 1.0!)")

assert abs(clipped_norm - 1.0) < 1e-5
print("\nSUCCESS: Circuit-breaker gradient clipping verified!")

Raw Gradient Vector:         [3.0, 4.0]
Raw Gradient Norm (Length):  5.0000 (Exceeds safety limit of 1.0!)

Clipped Gradient Vector:     [0.6, 0.8]
New Gradient Norm (Length):  1.0000 (Tamed to exactly 1.0!)

SUCCESS: Circuit-breaker gradient clipping verified!
